# Analisando dados do TCGA


##### 1. Importação das bibliotecas
##### 2. Importação dos dados do drive
###### 2.1. Montando o drive
###### 2.2. Importando o csv
##### 3. Pré processamento
###### 3.1. Verificando a descrição dos dados
###### 3.2. Identificando os tipos das colunas
###### 3.2.1. Verificando valores nulos
###### 3.2.2. Verificando os valores únicos de cada variável
###### 3.3. Verificando algumas medidas estatísticas
###### 3.4. Avaliando a quantidade de pessoas em cada classe
###### 3.5. Preenchimento dos dados faltantes
###### 3.6. Ajustando as colunas dummies
###### 3.7. Verificando as novas colunas
###### 3.8. Realizando a divisão do conjunto de dados entre treino e teste
###### 3.9. Seperando o conjunto de dados em treino e teste
##### 4. Balanceamento dos dados de treino
###### 4.1. SMOTE - criando instâncias sintáticas
###### 4.2. RandomUnderSamples - cortando instâncias
###### 4.3. Modelo híbrido, cortando instâncias da classe majoritária usando ClusterCentroids e depois aplicando o smote gerando instâncias para igualar Smote
###### 4.4. Modelo híbrido, cortando instâncias da classe majoritária usando RandomUnderSampler e depois aplicando o smote gerando instâncias para igualar Smote
###### 4.5. Mostrando os resultados
##### 5. Normalização
##### 6. Modelos de ML
###### 6.1. Random Forest
###### 6.2. Treinando o modelo Regressão logística
###### 6.3. Treinando o modelo KNN
###### 6.4. Treinando o modelo Árvore de decisão
###### 6.5. Teste com o SVM (SVC)
##### 7. Validação cruzada
##### 8. SHAP
##### 9. Curva de aprendizado

# 1. Importações necessárias

In [ ]:
print('Realizando algumas importações...',end=' ')
#Necessário para importar o csv ou xlsx
import pandas as pd
#Necessário para ignorar algumas mensagens de warning
import warnings
#Geração de gráficos
import matplotlib.pyplot as plt
#Geração de gráficos
import seaborn as sns
import time
#Cálculos matemáticos
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from imblearn import under_sampling, over_sampling
from imblearn.over_sampling import SMOTE
#Outra opção de Smote (gera na borda das classes)
from imblearn.over_sampling import BorderlineSMOTE
#Balanceamento
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, accuracy_score
from sklearn import metrics
#Identificar acodificação do arquivo
import chardet
#Para imputação de dados faltantes
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
#Modelo de classificação
from sklearn.ensemble import RandomForestClassifier
#Regressão logística
from sklearn.linear_model import LogisticRegression
#DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
#KNN
from sklearn.neighbors import KNeighborsClassifier
#SVM
from sklearn.svm import SVC
#Algumas configurações do pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.4f}'.format
#Necessário para curva de aprendizado
from sklearn.model_selection import learning_curve
# Necessário para validação cruzada
from sklearn.model_selection import cross_val_score, KFold
# Necessário para calcular métricas
from sklearn.metrics import classification_report
# Para gerar a matriz de confusão
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Para calcular a correlação
from sklearn.feature_selection import SelectKBest, chi2, f_classif
# Para curva ROC
from sklearn.metrics import roc_curve, roc_auc_score

print('[ok]')

# Configurações para tamanho das fontes dos gráficos

In [ ]:
plt.rc('font', size=14)  # Tamanho padrão para textos
plt.rc('axes', titlesize=16)  # Tamanho do título
plt.rc('axes', labelsize=14)  # Tamanho dos rótulos dos eixos
plt.rc('xtick', labelsize=12)  # Tamanho dos ticks no eixo X
plt.rc('ytick', labelsize=12)  # Tamanho dos ticks no eixo Y

# 2. Importação dos dados

### 2.2. Carregando os dados do drive...

In [ ]:
todos = True
#
print("Importando os dados ...",end=' ')
df_original = pd.read_excel('./dados.xlsx')
print("[ok]")
df_original.head()

## Contagem de instâncias do dataset

In [ ]:
df_original.shape

# 3. Pré-processamento

### 3.1 Verificando a descrição dos dados

In [ ]:
df_original.describe(include='all')

### 3.2. Identificando os tipos das colunas

#### 3.2.1. Verificando valores nulos

In [ ]:
#Verificando informações
df_original.info()

In [ ]:
#Verificando valores nulos
print(df_original.isnull().sum())

#### 3.2.2. Verificando os valores únicos de cada variável

In [ ]:
valores_unicos = []
for i in df_original.columns[0:43].tolist():
  print(i,':',len(df_original[i].astype(str).value_counts()))
  valores_unicos.append(len(df_original[i].astype(str).value_counts()))

### 3.3. Verificando algumas medidas estatísticas

In [ ]:
df_original.describe()

### 3.4. Avaliando a quantidade de pessoas em cada classe

In [ ]:
df_original.groupby(['sobrevida_curta_longa_5_anos']).size()

###### Fica muito desbalanceado para uma classe, o que pode deixar o modelo com vies de aprendizado

In [ ]:
#Montando um gráfico
#df_original.sobrevida_curta_loga_5_anos.value_counts().plot(kind='bar',title='Sobrevida 5 anos', color=['#219ebc','#023047'])

plt.rcParams['figure.figsize'] = [5.00,5.00]
plt.rcParams['figure.autolayout'] = True
df_original.sobrevida_curta_longa_5_anos.value_counts().plot(kind='bar', title='Sobrevida',color=['#1F77B4','#FF7F0E'])

### 3.5. Listando colunas para remover


In [ ]:
remover = [
           'days_to_death',
           'ethnicity', 
           'vital_status', 
           'year_of_birth', 
           'year_of_death', 
           'ajcc_clinical_m',
           'ajcc_clinical_n',
           'ajcc_clinical_t',
           'ajcc_staging_system_edition', 
           'classification_of_tumor', 
           'icd_10_code', 
           'last_known_disease_status', 
           'prior_malignancy', 
           'prior_treatment', 
           'progression_or_recurrence', 
           'year_of_diagnosis', 
           'treatment_intent_type', 
           'cause_of_death',
           'ethnicity',
           'race',
           'ajcc_clinical_stage',
           'ajcc_pathologic_stage',
           'morphology',
           'method_of_diagnosis',
           'therapeutic_agents',
           'treatment_or_therapy',
           'treatment_outcome',
           'treatment_type',
           'days_to_recurrence',
           'residual_disease',
           'tumor_grade'
           ]
df_original.drop(columns=remover, inplace=True)
#Verificando informações
df_original.info()

### 3.5. Etapa de imputação de dados faltantes

In [ ]:
print('Ajustando as colunas do dataset')

# Verifica se a coluna 'cause_of_death' existe no DataFrame
if 'cause_of_death' in df_original.columns:
    print('Preenchendo a coluna cause_of_death com o valor ', end=' ')

    # Preencher os valores nulos com o valor mais frequente
    df_original.loc[df_original['cause_of_death'].isna(), 'ethnicity'] = 'not_reported'
    # df_original['ethnicity'].fillna('not_reported', inplace=True)
    print('not_reported [ok]')
else:
    print("A coluna 'cause_of_death' não existe no DataFrame.")
##
#Verificando se a coluna 'ethnicity' existe
if 'ethnicity' in df_original.columns:
  print('Preenchendo a coluna ethnicity com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['ethnicity'].isna(), 'ethnicity'] = 'not_reported'
  #df_original['ethnicity'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ethnicity' não existe no DataFrame.")
##
#Verficando se a coluna 'gender' existe
if 'gender' in df_original.columns:
  print('Preenchendo a coluna gender com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['gender'].isna(), 'gender'] = 'not_reported'
  #df_original['gender'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'gender' não existe no DataFrame.")
##
#Verificando se a coluna 'race' exite
if 'race' in df_original.columns:
  print('Preenchendo a coluna race com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['race'].isna(), 'race'] = 'not_reported'
  #df_original['race'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'race' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_clinical_m' exite
if 'ajcc_clinical_m' in df_original.columns:
  print('Preenchendo a coluna ajcc_clinical_m com o valor ', end=' ')
  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_clinical_m'].isna(), 'ajcc_clinical_m'] = 'not_reported'
  #df_original['ajcc_clinical_m'].fillna('MX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_clinical_m' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_clinical_n' existe
if 'ajcc_clinical_n' in df_original.columns:
  print('Preenchendo a coluna ajcc_clinical_n com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_clinical_n'].isna(), 'ajcc_clinical_n'] = 'not_reported'
  #df_original['ajcc_clinical_n'].fillna('NX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_clinical_n' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_clinical_stage' existe
if 'ajcc_clinical_stage' in df_original.columns:
  print('Preenchendo a coluna ajcc_clinical_stage com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['ajcc_clinical_stage'].isna(), 'ajcc_clinical_stage'] = 'not_reported'
  #df_original['ajcc_clinical_stage'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_clinical_stage' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_clinical_t' existe
if 'ajcc_clinical_t' in df_original.columns:
  print('Preenchendo a coluna ajcc_clinical_t com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_clinical_t'].isna(), 'ajcc_clinical_t'] = 'not_reported'
  #df_original['ajcc_clinical_t'].fillna('TX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_clinical_t' não existe no DataFrame.")
##

#Verificando se a coluna ajcc_pathologic_m existe
if 'ajcc_pathologic_m' in df_original.columns:
  print('Preenchendo a coluna ajcc_pathologic_m com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_pathologic_m'].isna(), 'ajcc_pathologic_m'] = 'not_reported'
  #df_original['ajcc_pathologic_m'].fillna('MX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_pathologic_m' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_pathologic_n existe
if 'ajcc_pathologic_n' in df_original.columns:
  print('Preenchendo a coluna ajcc_pathologic_n com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_pathologic_n'].isna(), 'ajcc_pathologic_n'] = 'not_reported'
  #df_original['ajcc_pathologic_n'].fillna('NX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_pathologic_n' não existe no DataFrame.")
##

#Verificando se a coluna 'ajcc_pathologic_stage'
if 'ajcc_pathologic_stage' in df_original.columns:
  print('Preenchendo a coluna ajcc_pathologic_stage com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['ajcc_pathologic_stage'].isna(), 'ajcc_pathologic_stage'] = 'not_reported'
  #df_original['ajcc_pathologic_stage'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_pathologic_stage' não existe no DataFrame.")
##

#Verificando se a coluba 'ajcc_pathologic_t'
if 'ajcc_pathologic_t' in df_original.columns:
  print('Preenchendo a coluna ajcc_pathologic_t com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['ajcc_pathologic_t'].isna(), 'ajcc_pathologic_t'] = 'not_reported'
  #df_original['ajcc_pathologic_t'].fillna('TX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'ajcc_pathologic_t' não existe no DataFrame.")
##

#Verificando se a coluna 'classification_of_tumor' existe
if 'classification_of_tumor' in df_original.columns:
  print('Preenchendo a coluna classification_of_tumor com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['classification_of_tumor'].isna(), 'classification_of_tumor'] = 'not_reported'
  #df_original['classification_of_tumor'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'classification_of_tumor' não existe no DataFrame.")
##

#Verificando se a coluna 'method_of_diagnosis' existe
if 'method_of_diagnosis' in df_original.columns:
  print('Preenchendo a coluna method_of_diagnosis com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['method_of_diagnosis'].isna(), 'method_of_diagnosis'] = 'not_reported'
  #df_original['method_of_diagnosis'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'method_of_diagnosis' não existe no DataFrame.")
##

#Verificando se a coluna 'morphology' existe
if 'morphology' in df_original.columns:
  print('Preenchendo a coluna morphology com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['morphology'].isna(), 'morphology'] = 'not_reported'
  #df_original['morphology'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'morphology' não existe no DataFrame.")
##

#Verificando se a coluna 'morphology' existe
if 'primary_diagnosis' in df_original.columns:
  print('Preenchendo a coluna primary_diagnosis com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['primary_diagnosis'].isna(), 'primary_diagnosis'] = 'not_reported'
  #df_original['morphology'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'primary_diagnosis' não existe no DataFrame.")
##

#Verificando se a coluna 'residual_disease'
if 'residual_disease' in df_original.columns:
  print('Preenchendo a coluna residual_disease com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['residual_disease'].isna(), 'residual_disease'] = 'not_reported'
  #df_original['residual_disease'].fillna('RX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'residual_disease' não existe no DataFrame.")
##

#Verificando se a coluna 'site_of_resection_or_biopsy' existe
if 'site_of_resection_or_biopsy' in df_original.columns:
  print('Preenchendo a coluna site_of_resection_or_biopsy com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['site_of_resection_or_biopsy'].isna(), 'site_of_resection_or_biopsy'] = 'not_reported'
  #df_original['site_of_resection_or_biopsy'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'site_of_resection_or_biopsy' não existe no DataFrame.")
##

#Verificando se a coluna 'tissue_or_organ_of_origin'  existe
if 'tissue_or_organ_of_origin' in df_original.columns:
  print('Preenchendo a coluna tissue_or_organ_of_origin com o valor ', end=' ')

  # Preencher os valores nulos com o valor mais frequente
  df_original.loc[df_original['tissue_or_organ_of_origin'].isna(), 'tissue_or_organ_of_origin'] = 'not_reported'
  #df_original['tissue_or_organ_of_origin'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'tissue_or_organ_of_origin' não existe no DataFrame.")
##

#Verificando se a coluna tumor_grade existe
if 'tumor_grade' in df_original.columns:
  print('Preenchendo a coluna tumor_grade com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['tumor_grade'].isna(), 'tumor_grade'] = 'not_reported'
  #df_original['tumor_grade'].fillna('RX', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'tumor_grade' não existe no DataFrame.")
##

#Verficando se a coluna 'therapeutic_agents' existe
if 'therapeutic_agents' in df_original.columns:
  print('Preenchendo a coluna therapeutic_agents com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['therapeutic_agents'].isna(), 'therapeutic_agents'] = 'not_reported'
  #df_original['therapeutic_agents'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'therapeutic_agents' não existe no DataFrame.")
##

#Verificando se a coluna treatment_outcome existe
if 'treatment_outcome' in df_original.columns:
  print('Preenchendo a coluna treatment_outcome com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['treatment_outcome'].isna(), 'treatment_outcome'] = 'not_reported'
  #df_original['treatment_outcome'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'treatment_outcome' não existe no DataFrame.")
##

#Verificando se a coluna treatment_type existe
if 'treatment_type' in df_original.columns:
  print('Preenchendo a coluna treatment_type com o valor ', end=' ')

  # Preencher os valores nulos com not_reported
  df_original.loc[df_original['treatment_type'].isna(), 'treatment_type'] = 'not_reported'
  #df_original['treatment_type'].fillna('not_reported', inplace=True)

  print('not_reported [ok]')
else:
  print("A coluna 'treatment_type' não existe no DataFrame.")
##

#Verificando se a coluna age_at_index_calc existe
if 'age_at_index_calc' in df_original.columns:
  print('Preenchendo a coluna numérica age_at_index_calc com o valor ', end=' ')

  #Calculando a mediana para preenchimento
  mediana = df_original['age_at_index_calc'].median()

  #Preenchendo os nulos com a mediana calculada
  df_original.loc[df_original['age_at_index_calc'].isna(), 'age_at_index_calc'] = mediana
  #df_original['age_at_index_calc'].fillna(mediana, inplace=True)

  print(mediana,'[ok]')
else:
  print("A coluna 'age_at_index_calc' não existe no DataFrame.")
##

#Verificando se a coluna days_to_last_follow_up existe
if 'days_to_last_follow_up' in df_original.columns:

  print('Preenchendo a coluna numérica days_to_last_follow_up com o valor ', end=' ')

  #Calculando a mediana para preenchimento
  mediana = df_original['days_to_last_follow_up'].median();
  #Preenchendo os nulos com a mediana calculada
  df_original.loc[df_original['days_to_last_follow_up'].isna(), 'days_to_last_follow_up'] = mediana
  #df_original['days_to_last_follow_up'].fillna(mediana, inplace=True)

  print(mediana,'[ok]')
else:
  print("A coluna 'days_to_last_follow_up' não existe no DataFrame.")
##

#Verificando se a coluna days_to_recurrence existe
if 'days_to_recurrence' in df_original.columns:
  print('Preenchendo a coluna numérica days_to_recurrence com o valor ', end=' ')

  #Calculando a mediana para preenchimento
  mediana = df_original['days_to_recurrence'].median();

  #Preenchendo os nulos com a mediana calculada
  df_original.loc[df_original['days_to_recurrence'].isna(), 'days_to_recurrence'] = mediana
  #df_original['days_to_recurrence'].fillna(mediana, inplace=True)

  print(mediana,'[ok]')
else:
  print("A coluna 'days_to_recurrence' não existe no DataFrame.")
##

#Verificando se a coluna synchronous_malignancy existe
if 'synchronous_malignancy' in df_original.columns:
  print('Preenchendo a coluna binária synchronous_malignancy com o valor ', end=' ')

  #Preenchendo a coluna synchronous_malignancy com No
  df_original.loc[df_original['synchronous_malignancy'].isna(), 'synchronous_malignancy'] = 'No'
  #df_original['synchronous_malignancy'].fillna('No', inplace=True)

  print('No [ok]')
else:
  print("A coluna 'synchronous_malignancy' não existe no DataFrame.")
##

#Verificando se a coluna treatment_or_therapy existe
if 'treatment_or_therapy' in df_original.columns:
  print('Preenchendo a coluna binária treatment_or_therapy com o valor ', end=' ')

  #Preenchendo a coluna synchronous_malignancy com No
  df_original.loc[df_original['treatment_or_therapy'].isna(), 'treatment_or_therapy'] = 'No'
  #df_original['treatment_or_therapy'].fillna('No', inplace=True)

  print('No [ok]')
else:
  print("A coluna 'treatment_or_therapy' não existe no DataFrame.")
##


In [ ]:
#Verificando valores nulos
print(df_original.isnull().sum())

In [ ]:
# Lista de atributos
atributos = [
    "age_at_index_calc",
    "sobrevida_curta_longa_5_anos",
    "gender",
    "ajcc_pathologic_m",
    "ajcc_pathologic_n",
    "ajcc_pathologic_t",
    "days_to_last_follow_up",
    "primary_diagnosis",
    "site_of_resection_or_biopsy",
    "synchronous_malignancy",
    "tissue_or_organ_of_origin"
]

# Iterar sobre os atributos e exibir valores únicos e contagens
for atributo in atributos:
    print(f"\nAnalisando a coluna: {atributo}")
    print(df_original[atributo].value_counts())  # Valores únicos e suas contagens
    print(f"\nTotal de valores únicos na coluna '{atributo}': {df_original[atributo].nunique()}")

### 3.6. Ajustando colunas dummies (colunas categóricas nominais), ajustando as colunas categóricas ordinais

In [ ]:
if 'cause_of_death' in df_original.columns:
  print('Ajustando a coluna cause_of_death', end=' ')

  # Criar colunas dummy usando one-hot encoding
  # .rename(columns=lambda x: x.replace(' ', '_')) faz com que os valores que possuam espaço na coluna tenham estes espaços trocados por _
  # .rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
  encoded_columns = pd.get_dummies(df_original['cause_of_death'], prefix='cause_of_death').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'cause_of_death'
  df_original.drop('cause_of_death', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'cause_of_death' não existe no DataFrame.")
##

if 'ethnicity' in df_original.columns:
  print('Ajustando a coluna ethnicity', end=' ')

  # Criar colunas dummy usando one-hot encoding
  # .rename(columns=lambda x: x.replace(' ', '_')) faz com que os valores que possuam espaço na coluna tenham estes espaços trocados por _
  encoded_columns = pd.get_dummies(df_original['ethnicity'], prefix='ethnicity').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'ethnicity'
  df_original.drop('ethnicity', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'ethnicity' não existe no DataFrame.")
##

if 'gender' in df_original.columns:
  print('Ajustando a coluna gender', end=' ')
  ##
  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['gender'], prefix='gender').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'gender'
  df_original.drop('gender', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'gender' não existe no DataFrame.")
##

if 'race' in df_original.columns:
  print('Ajustando a coluna race', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['race'], prefix='race').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'race'
  df_original.drop('race', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'race' não existe no DataFrame.")
##

if 'ajcc_clinical_m' in df_original.columns:
  print('Ajustando a coluna ajcc_clinical_m', end=' ')

  #ordem_categorias = {'M0':0, 'M1':1, 'not_reported':2, 'M1a':3, 'M1b':4,'MX':5}
  ordem_categorias = {'M0':0, 'MX':1, 'not_reported':2, 'M1':3, 'M1a':4, 'M1b':5}

  # Mapear os valores da coluna para números
  df_original['ajcc_clinical_m'] = df_original['ajcc_clinical_m'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_clinical_m' não existe no DataFrame.")
##

if 'ajcc_clinical_n' in df_original.columns:
  print('Ajustando a coluna ajcc_clinical_n', end=' ')

  if todos:
    #ordem_categorias = {'N0':0, 'N1':1, 'not_reported':2, 'N2':3, 'N3':4, 'NX':5}
    ordem_categorias = {'N0':0, 'NX':1, 'not_reported':2, 'N1':3, 'N2':4, 'N3':5}
  else:
    #ordem_categorias = {'N0':0, 'N2':1, 'not_reported':2, 'N3':3, 'NX':4}
    ordem_categorias = {'N0':0, 'NX':1, 'not_reported':2, 'N2':3, 'N3':4}

  # Mapear os valores da coluna para números
  df_original['ajcc_clinical_n'] = df_original['ajcc_clinical_n'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_clinical_n' não existe no DataFrame.")
##

if 'ajcc_clinical_stage' in df_original.columns:
  print('Ajustando a coluna ajcc_clinical_stage', end=' ')
  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['ajcc_clinical_stage'], prefix='ajcc_clinical_stage').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'ajcc_clinical_stage'
  df_original.drop('ajcc_clinical_stage', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'ajcc_clinical_stage' não existe no DataFrame.")
##

if 'ajcc_clinical_t' in df_original.columns:
  print('Ajustando a coluna ajcc_clinical_t', end=' ')
  if todos:
    ordem_categorias = {'T1':0, 'TX':1, 'T1a':2, 'not_reported':3,'T2':4, 'T2a':5, 'T2b':6, 'T3':7, 'T4':8}
  else:
    ordem_categorias = {'T1':0, 'TX':1, 'T2':2, 'not_reported':3, 'T2a':4, 'T2b':5, 'T3':6, 'T4':7}

  # Mapear os valores da coluna para números
  df_original['ajcc_clinical_t'] = df_original['ajcc_clinical_t'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_clinical_t' não existe no DataFrame.")
##

if 'ajcc_pathologic_m' in df_original.columns:
  print('Ajustando a coluna ajcc_pathologic_m', end=' ')

  ordem_categorias = {'M0':0, 'MX':1, 'not_reported':2, 'M1':3, 'M1a':4, 'M1b':5}

  # Mapear os valores da coluna para números
  df_original['ajcc_pathologic_m'] = df_original['ajcc_pathologic_m'].map(ordem_categorias)
  print('[ok]')
else:
  print("A coluna 'ajcc_pathologic_m' não existe no DataFrame.")
##

if 'ajcc_pathologic_n' in df_original.columns:
  print('Ajustando a coluna ajcc_pathologic_n', end=' ')

  # Preencher os valores nulos com not_reported

  ordem_categorias = {'N0':0, 'NX':1, 'N1':2, 'not_reported':3, 'N2':4, 'N3':5}

  # Mapear os valores da coluna para números
  df_original['ajcc_pathologic_n'] = df_original['ajcc_pathologic_n'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_pathologic_n' não existe no DataFrame.")
##

if 'ajcc_pathologic_stage' in df_original.columns:
  print('Ajustando a coluna ajcc_pathologic_stage', end=' ')

  ordem_categorias = {'Stage 1':0, 'Stage IA':1, 'Stage IA2':2, 'Stage IA3':3, 'Stage IB':4, 'not_reported':5, 'Stage II':6, 'Stage IIA':7, 'Stage IIB':8, 'Stage III':9, 'Stage IIA':10, 'Stage IIB':11}

  # Mapear os valores da coluna para números
  df_original['ajcc_pathologic_stage'] = df_original['ajcc_pathologic_stage'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_pathologic_stage' não existe no DataFrame.")
##

if 'ajcc_pathologic_t' in df_original.columns:
  print('Ajustando a coluna ajcc_pathologic_t', end=' ')

  ordem_categorias = {'T1':0, 'TX':1, 'T1a':2, 'T1b':3, 'T1c':4, 'not_reported':5, 'T2':6, 'T2a':7, 'T2b':8, 'T3':9, 'T3a':10, 'T4':11}

  # Mapear os valores da coluna para números
  df_original['ajcc_pathologic_t'] = df_original['ajcc_pathologic_t'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'ajcc_pathologic_t' não existe no DataFrame.")
##

if 'classification_of_tumor' in df_original.columns:
  print('Ajustando a coluna classification_of_tumor', end=' ')

  ordem_categorias = {'not reported':0, 'primary':1, 'Prior primary':2, 'metastasis':3}

  # Mapear os valores da coluna para números
  df_original['classification_of_tumor'] = df_original['classification_of_tumor'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'classification_of_tumor' não existe no DataFrame.")
##

if 'method_of_diagnosis' in df_original.columns:
  print('Ajustando a coluna method_of_diagnosis', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['method_of_diagnosis'], prefix='method_of_diagnosis').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'method_of_diagnosis'
  df_original.drop('method_of_diagnosis', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'method_of_diagnosis' não existe no DataFrame.")
##

if 'morphology' in df_original.columns:
  print('Ajustando a coluna morphology', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['morphology'], prefix='morphology').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'morphology'
  df_original.drop('morphology', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'morphology' não existe no DataFrame.")
##

if 'primary_diagnosis' in df_original.columns:
  print('Ajustando a coluna primary_diagnosis', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['primary_diagnosis'], prefix='primary_diagnosis').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'morphology'
  df_original.drop('primary_diagnosis', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'morphology' não existe no DataFrame.")
##

if 'residual_disease' in df_original.columns:
  print('Ajustando a coluna residual_disease', end=' ')

  ordem_categorias = {'R0':0, 'RX':1, 'not_reported':2, 'R1':3, 'R2':4}

  # Mapear os valores da coluna para números
  df_original['residual_disease'] = df_original['residual_disease'].map(ordem_categorias)

  print('[ok]')
else:
  print("A coluna 'residual_disease' não existe no DataFrame.")
##

if 'site_of_resection_or_biopsy' in df_original.columns:
  print('Ajustando a coluna site_of_resection_or_biopsy', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['site_of_resection_or_biopsy'], prefix='site_of_resection_or_biopsy').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'site_of_resection_or_biopsy'
  df_original.drop('site_of_resection_or_biopsy', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'site_of_resection_or_biopsy' não existe no DataFrame.")
##

if 'tissue_or_organ_of_origin' in df_original.columns:
  print('Ajustando a coluna tissue_or_organ_of_origin', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['tissue_or_organ_of_origin'], prefix='tissue_or_organ_of_origin').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))
  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'tissue_or_organ_of_origin'
  df_original.drop('tissue_or_organ_of_origin', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'tissue_or_organ_of_origin' não existe no DataFrame.")
##

if 'tumor_grade' in df_original.columns:
  print('Ajustando a coluna tumor_grade', end=' ')

  ordem_categorias = {'G1':0, 'GX':1, 'not_reported':2, 'G2':3, 'G3':4}

  # Mapear os valores da coluna para números
  df_original['tumor_grade'] = df_original['tumor_grade'].map(ordem_categorias)

  # Remover a coluna original 'therapeutic_agents'
  df_original.drop('tumor_grade', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'tumor_grade' não existe no DataFrame.")
##

if 'therapeutic_agents' in df_original.columns:
  print('Ajustando a coluna therapeutic_agents', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['therapeutic_agents'], prefix='therapeutic_agents').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'therapeutic_agents'
  df_original.drop('therapeutic_agents', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'therapeutic_agents' não existe no DataFrame.")
##

if 'treatment_outcome' in df_original.columns:
  print('Ajustando a coluna treatment_outcome', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['treatment_outcome'], prefix='treatment_outcome').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'treatment_outcome'
  df_original.drop('treatment_outcome', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'treatment_outcome' não existe no DataFrame.")
##

if 'treatment_type' in df_original.columns:
  print('Ajustando a coluna treatment_type', end=' ')

  # Criar colunas dummy usando one-hot encoding
  encoded_columns = pd.get_dummies(df_original['treatment_type'], prefix='treatment_type').rename(columns=lambda x: x.replace(' ', '_').replace('(', '').replace(')', '').replace(',', ''))

  # Concatenar as colunas dummy com o dataset original
  df_original = pd.concat([df_original, encoded_columns], axis=1)

  # Remover a coluna original 'treatment_type'
  df_original.drop('treatment_type', axis=1, inplace=True)

  print('[ok]')
else:
  print("A coluna 'treatment_type' não existe no DataFrame.")
##

if 'synchronous_malignancy' in df_original.columns:
  print('Ajustando a coluna binária synchronous_malignancy', end=' ')

  # Mapeando os valores 'Yes' para 1 e 'No' para 0
  df_original['synchronous_malignancy'] = df_original['synchronous_malignancy'].map({'Yes': 1, 'No': 0})

  print('[ok]')
else:
  print("A coluna 'synchronous_malignancy' não existe no DataFrame.")
##

if 'treatment_or_therapy' in df_original.columns:
  print('Ajustando a coluna binária treatment_or_therapy', end=' ')

  # Mapeando os valores 'Yes' para 1 e 'No' para 0
  df_original['treatment_or_therapy'] = df_original['treatment_or_therapy'].map({'Yes': 1, 'No': 0})

  print('[ok]')
else:
  print("A coluna 'treatment_or_therapy' não existe no DataFrame.")
##

##### ajustando a colunas alvo

In [ ]:
print('Ajustando as colunas de classe_sobrevida', end=' ')

# Mapeando os valores 'Longa' para 1 e 'Curta' para 0
df_original['sobrevida_curta_longa_5_anos'] = df_original['sobrevida_curta_longa_5_anos'].map({'longa': 1, 'curta': 0})

print('[ok]')

### 3.7. Verificando as novas colunas

In [ ]:
#Verificando valores nulos
print(df_original.isnull().sum())

---

### 3.8. Realizando separação das colunas de treinamento (x) e da coluna alvo (y)

In [ ]:
print('Separando as colunas para análise e a coluna alvo', end=' ')

# Definindo X como todas as colunas exceto 'sobrevida_curta_longa_5_anos'
x = df_original.drop(columns=['sobrevida_curta_longa_5_anos'])

y = df_original['sobrevida_curta_longa_5_anos']
#y = df_original['sobrevida_curta_loga_2_anos']
print('[ok]')

In [ ]:
df_original.head(n=10)

###### Deixar 30% para treino e o restante para teste

### 3.9. Seperando o conjunto de dados em treino e teste

In [ ]:
print('Separando o dataset em treino e teste =0.4 deixa 40% para teste e 60% para treinamento')
# Dividindo o dataset
x_treino, x_test, y_treino, y_test = train_test_split(x, y, test_size=0.4, random_state=42, stratify=y)
print('Instâncias de treino', x_treino.shape)
print('Instâncias de teste', x_test.shape)


#Contagem de instâncias de teste agrupando por classe
classes_treino, counts_treino = np.unique(y_treino, return_counts=True)
classes_teste, counts_teste = np.unique(y_test, return_counts=True)

# Exibir resultados
print("Instâncias de treino por classe:")
for classe, count in zip(classes_treino, counts_treino):
    print(f"Classe {classe}: {count}")

print("\nInstâncias de teste por classe:")
for classe, count in zip(classes_teste, counts_teste):
    print(f"Classe {classe}: {count}")

print('[ok]')
#Usado para o cross validation
y_treino_n_balanceado = y_treino

# 4. Balanceando os dados de treino criando instâncias sintéticas (escolha entre uma das opções)


### 4.1. SMOTE - criando instâncias sintáticas



In [ ]:
# seed para produzir o mesmo resultado
seed = 100

# Criando um balanceador SMOTE
balanceador = SMOTE(random_state=seed)

# Aplicando o balanceador
x_res, y_res = balanceador.fit_resample(x_treino, y_treino)

print(x_res.shape)
print(y_res.shape)
y_treino = y_res

### 4.1.1. BorderSMOTE - criando instâncias sintáticas

In [ ]:
# Seed para reprodutibilidade
seed = 100

# Criando um balanceador Borderline-SMOTE
balanceador = BorderlineSMOTE(random_state=seed, kind="borderline-1")  # Pode usar 'borderline-2' para outra abordagem

# Aplicando o balanceador
x_res, y_res = balanceador.fit_resample(x_treino, y_treino)

# Resultados
print(x_res.shape)
print(y_res.shape)

# Atualizando y_treino
y_treino = y_res

### 4.2. RandomUnderSamples - cortando instâncias

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# Definindo a semente para reprodutibilidade
seed = 100

# Criando um balanceador RandomUnderSampler
balanceador = RandomUnderSampler(random_state=seed)

# Aplicando o balanceador
x_res, y_res = balanceador.fit_resample(x_treino, y_treino)

print(x_res.shape)
print(y_res.shape)

#Mantendo a compatibilidade, os dados são esperados dentro de y_treino
y_treino = y_res

### 4.3. Modelo híbrido, cortando instâncias da classe majoritária e depois aplicando o smote gerando instâncias para igualar (OPÇÃO APLICADA NOS TESTES)

In [ ]:
from imblearn.under_sampling import ClusterCentroids
from collections import Counter

seed = 100

# Criando o undersample com Cluster Centroids
undersample = ClusterCentroids(sampling_strategy=0.7, random_state=seed)

# Aplicando o Cluster Centroids
x_unsers, y_unsers = undersample.fit_resample(x_treino, y_treino)

# Verificando o novo balanço das classes
print(f"Distribuição original: {Counter(y_treino)}")
print(f"Distribuição após Cluster Centroids: {Counter(y_unsers)}")

# Criando o smote para gerar 30% das instâncias
smote = SMOTE(random_state=seed)
# Aplicando o smote
x_res, y_res = smote.fit_resample(x_unsers, y_unsers )

print(x_res.shape)
print(y_res.shape)

# Mantendo a compatibilidade
y_treino = y_res


# Verificando o novo balanço das classes
from collections import Counter
print(f"Distribuição original: {Counter(y_treino)}")
print(f"Distribuição após SMOTE: {Counter(y_res)}")

### 4.4. Modelo híbrido, cortando instâncias da classe majoritária usando RandomUnderSampler e depois aplicando o smote gerando instâncias para igualar Smote

In [ ]:
from collections import Counter

seed = 100

# Criando o undersample com Cluster Centroids
undersample = RandomUnderSampler(sampling_strategy=0.7, random_state=seed)

# Aplicando o Cluster Centroids
x_unsers, y_unsers = undersample.fit_resample(x_treino, y_treino)

# Verificando o novo balanço das classes
print(f"Distribuição original: {Counter(y_treino)}")
print(f"Distribuição após Cluster Centroids: {Counter(y_unsers)}")

# Criando o smote para gerar 30% das instâncias
smote = SMOTE(random_state=seed)
# Aplicando o smote
x_res, y_res = smote.fit_resample(x_unsers, y_unsers )

print(x_res.shape)
print(y_res.shape)

# Mantendo a compatibilidade
y_treino = y_res


# Verificando o novo balanço das classes
from collections import Counter
print(f"Distribuição original: {Counter(y_treino)}")
print(f"Distribuição após SMOTE: {Counter(y_res)}")

### 4.5. Mostrando os resultados

In [ ]:
plt.rcParams['figure.figsize'] = [5.00,5.00]
plt.rcParams['figure.autolayout'] = True
y_treino.value_counts().plot(kind='bar', title='Sobrevida',color=['#1F77B4','#FF7F0E'])

In [ ]:
from collections import Counter
# Contagem das classes
count_original = Counter(y_treino_n_balanceado)
count_balanced = Counter(y_treino)

# Criando o gráfico
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# Antes do balanceamento
sns.barplot(
    x=list(count_original.keys()),
    y=list(count_original.values()),
    hue=list(count_original.keys()),  # Associa o x ao hue
    ax=axes[0],
    palette="muted",
    legend=False  # Remove a legenda
)
axes[0].set_title("Distribuição Antes do Balanceamento")
axes[0].set_xlabel("Classes")
axes[0].set_ylabel("Instâncias")

# Depois do balanceamento
sns.barplot(
    x=list(count_balanced.keys()),
    y=list(count_balanced.values()),
    hue=list(count_balanced.keys()),  # Associa o x ao hue
    ax=axes[1],
    palette="muted",
    legend=False  # Remove a legenda
)
axes[1].set_title("Distribuição Após o Balanceamento")
axes[1].set_xlabel("Classes")

plt.tight_layout()
plt.show()

# 5. Normalização

In [ ]:
# Criar o escalonador e ajustá-lo apenas nos dados de treino balanceados
normalizador = MinMaxScaler()
x_treino_norm = normalizador.fit_transform(x_res)  # Ajusta e transforma o treino balanceado

# Aplicar a mesma transformação ao conjunto de teste
x_teste_norm = normalizador.transform(x_test)  # Apenas transforma o teste

# Normalizando o conjunto de dados não balanceado
x_treino_norm_n_balanceado = normalizador.fit_transform(x_treino)  # Ajusta e transforma o treino não balanceado

# Aplicar a mesma transformação ao conjunto de teste não balanceado
x_teste_norm_n_balanceado = normalizador.transform(x_test)  # Apenas transforma o teste

#### Teste com wrapper iterativos

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.ensemble import RandomForestClassifier

x_treino_norm_df = pd.DataFrame(x_treino_norm, columns=x.columns)
y_treino_df = pd.Series(y_treino)

# Criar o seletor sequencial
sfs = SFS(RandomForestClassifier(),
          k_features=10,  # Número de características desejadas
          forward=True,   # Seleção forward
          scoring='accuracy',  # Métrica usada
          cv=5)           # Validação cruzada

# Ajustar o modelo com os dados normalizados
sfs.fit(x_treino_norm_df, y_treino_df)



# Obter os nomes das colunas selecionadas
selected_features = sfs.k_feature_names_

# Exibir como uma lista de colunas
print("Colunas Selecionadas:", list(selected_features))

# Modelos de AM

### 6.1. Random Forest

In [ ]:
#Calculando os melhores parâmetros

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Supomos que você já tem X_train, X_test, y_train, y_test

# Defina o modelo base
rf = RandomForestClassifier()

# Defina os parâmetros para busca
param_grid = {
    'n_estimators': [50, 100, 200, 300, 500],  # Adicionados 300 e 500
    'max_depth': [None, 10, 20, 30, 50],      # Adicionado 50
    'min_samples_split': [2, 5, 10, 15],      # Adicionado 15
    'min_samples_leaf': [1, 2, 4, 8],         # Novo parâmetro
    'bootstrap': [True, False]                # Novo parâmetro
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,             # Modelo base
    param_grid=param_grid,    # Conjunto de hiperparâmetros
    scoring='f1_weighted',    # Métrica a ser otimizada (pode ser outra, como recall ou precisão)
    cv=5,                     # Validação cruzada com 5 folds
    verbose=2,                # Nível de detalhamento na saída
    n_jobs=-1                 # Usa todos os núcleos disponíveis
)

# Executa a busca
grid_search.fit(x_treino_norm, y_treino)

# Melhor combinação de parâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor pontuação obtida
print("Melhor pontuação de validação cruzada:", grid_search.best_score_)

# Avalie o modelo no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_tunning = best_model.predict(x_teste_norm)

print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred_tunning))


In [ ]:
# Criando o classificador com RandomForest

#Parámetros calculados pelo 
clf = RandomForestClassifier(
    bootstrap = False,
    #criterion='entropy',
    n_estimators=100,        # Usar n_estimators = 100, como nos melhores parâmetros encontrados
    max_depth=50,            # Ajustando max_depth para 30
    #max_features='sqrt',     # Mantendo max_features como 'sqrt'
    min_samples_leaf = 1,
    min_samples_split=5,     # Ajustando min_samples_split para 5
    n_jobs=8,                # Mantendo n_jobs como 8 para paralelismo
    #class_weight='balanced',  # Mantendo o balanceamento de classes
    random_state=42
)

# Construção do modelo
clf = clf.fit(x_treino_norm, y_treino)
print('[ok]')

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

#modelo = RandomForestClassifier(random_state=42)
rfe = RFE(estimator=clf, n_features_to_select=10)
rfe.fit(x_treino_norm, y_treino)
variaveis_selecionadas = x.columns[rfe.support_]
print("Variáveis Selecionadas:", list(variaveis_selecionadas))

In [ ]:
# exibindo a importência de cada variável no modelo preditivo
plt.rcParams['figure.figsize'] = [10.00,30.00]
plt.rcParams['figure.autolayout'] = True
#Importâncias para gráfico geral
importancias_rf = clf.feature_importances_
importances = pd.Series(data=clf.feature_importances_, index=x.columns)
importances = importances.sort_values(ascending = False)
sns.barplot(x=importances, y=importances.index, orient='h').set_title('Importância de cada variável')
plt.show()

In [ ]:
#Apresentando o percentual de importância de cada variável
importances.sort_values(ascending = False)

In [ ]:
#Simulando os dados de teste
scores= clf.score(x_treino_norm, y_treino)
scores

#### Simulando com os dados de teste

In [ ]:
print('Realizando previsões com o Random Forest',end=' ')
#Realizando a previsão com os dados de teste
#Realizando as previsões
y_pred = clf.predict(x_teste_norm)

#Para o conjunto de treino
y_pred_treino = clf.predict(x_treino_norm)
print('[ok]')

In [ ]:
#Gerando a matriz de confusão do algoritmo

print(confusion_matrix(y_test, y_pred))

#### Calculando métricas

In [ ]:
#Calculado métricas de avaliação do modelo
print('Resultados para Random Forest')

print("Relatório de Classificação - Treino")
print(classification_report(y_treino, y_pred_treino, digits=4))

print("Relatório de Classificação - Teste")
print(classification_report(y_test, y_pred, digits=4))

print('[ok]')
# Relatórios de classificação
relatorio_treino = classification_report(y_treino, y_pred_treino, output_dict=True)
relatorio_teste = classification_report(y_test, y_pred, output_dict=True)

# Transformar em DataFrames
df_treino = pd.DataFrame(relatorio_treino).transpose()
df_teste = pd.DataFrame(relatorio_teste).transpose()

# Remover a linha de 'accuracy' do corpo principal para exibi-la separadamente
acuracia_treino = df_treino.loc['accuracy', 'f1-score']
acuracia_teste = df_teste.loc['accuracy', 'f1-score']

df_treino = df_treino.drop('accuracy')
df_teste = df_teste.drop('accuracy')

# Exibir os resultados formatados
print('=== Resultados para Random Forest ===\n')

print('Relatório de Classificação - Treino')
print(f'Acurácia: {acuracia_treino:.4f}')
display(df_treino.style.format(precision=4).background_gradient(cmap='Blues'))

print('\nRelatório de Classificação - Teste')
print(f'Acurácia: {acuracia_teste:.4f}')
display(df_teste.style.format(precision=4).background_gradient(cmap='Greens'))

print('[ok]')

In [ ]:
cm = confusion_matrix(y_test,y_pred)
print(cm)

#### Calculando o G-Mean

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Calculando sensibilidade e especificidade
sensitivity = tp / (tp + fn)  # Recall para a classe positiva
specificity = tn / (tn + fp)  # Recall para a classe negativa

# Calculando o G-Mean
g_mean = np.sqrt(sensitivity * specificity)

print(f"Sensibilidade: {sensitivity:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"G-Mean: {g_mean:.4f}")

In [ ]:
#Configure...
portugues = True
nome_algoritmo = 'Random Forest'

#Outra opção de matriz de confusão
pd.options.display.float_format = '{:.2f}'.format

plt.rcParams["figure.figsize"] = [5.00,5.00]
plt.rcParams["figure.autolayout"] = True

f, ax = plt.subplots(figsize=(5,5))

if portugues:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Curta","Longa"],yticklabels=["Curta","Longa"])
  ax.set_xlabel('Valor predito')
  ax.set_ylabel('Valor real')
  #titulo = 'Previsões de sobrevida longa (1) ou curta (0) '+nome_algoritmo
  titulo = nome_algoritmo
  plt.title(titulo)
else:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Short","Long"],yticklabels=["Short","Long"])
  ax.set_xlabel('Predicted value')
  ax.set_ylabel('True value')
  titulo = 'Predictions of Long (1) or Short (0) Survival '+nome_algoritmo
  plt.title(titulo)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tick_params(axis='both',which='major', labelbottom = False, top = True, labeltop = True)
plt.show()

In [ ]:
# Probabilidades previstas para a classe positiva (Classe 1)
y_pred_proba = clf.predict_proba(x_teste_norm)[:, 1]
# Obter valores de FPR (False Positive Rate), TPR (True Positive Rate) e thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Calcular a área sob a curva (AUC)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {auc_score:.4f}")
#Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label="Random Guess")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curva ROC Random Forest")
plt.legend(loc="lower right")
plt.grid()
plt.show()

### 6.2. Treinando o modelo Regressão logística

In [ ]:
# Verificando os parámetros para regressão lógistica
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Supomos que você já tem X_train, X_test, y_train, y_test

# Defina o modelo base
lr = LogisticRegression(random_state=42, solver='liblinear')

# Defina os parâmetros para busca
param_grid = {
    'penalty': ['l1', 'l2'],          # Tipo de regularização
    'C': [0.01, 0.1, 1, 10, 100],    # Inverso da força de regularização
    'class_weight': [None, 'balanced'],  # Lida com classes desbalanceadas
    'solver': ['liblinear', 'saga']  # Solvers compatíveis com l1 e l2
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=lr,              # Modelo base
    param_grid=param_grid,      # Conjunto de hiperparâmetros
    scoring='f1_weighted',      # Métrica a ser otimizada
    cv=5,                       # Validação cruzada com 5 folds
    verbose=2,                  # Nível de detalhamento na saída
    n_jobs=-1                   # Usa todos os núcleos disponíveis
)

# Executa a busca
grid_search.fit(x_treino_norm, y_treino)

# Melhor combinação de parâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor pontuação obtida
print("Melhor pontuação de validação cruzada:", grid_search.best_score_)

# Avalie o modelo no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_tunning = best_model.predict(x_teste_norm)

print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred_tunning))

In [ ]:
print('Treinando o modelo de regressão logística ...',end='')

# Ajustando o modelo com os parâmetros fornecidos
# Ajustando o modelo com os melhores parâmetros encontrados
reglog = LogisticRegression(
    penalty='l2',          # Tipo de regularização Lasso
    C=10,                   # Inverso da força de regularização
    solver='liblinear',         # Solver compatível com regularização L1
    max_iter=1000,         # Número máximo de iterações
    class_weight='balanced',     # Sem ajuste automático de peso para classes
    random_state=42        # Define a semente para reprodutibilidade
)

reglog.fit(x_treino_norm, y_treino)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

#modelo = RandomForestClassifier(random_state=42)
rfe = RFE(estimator=reglog, n_features_to_select=10)
rfe.fit(x_treino_norm, y_treino)
variaveis_selecionadas = x.columns[rfe.support_]
print("Variáveis Selecionadas:", list(variaveis_selecionadas))

#### Realizando previsões

In [ ]:
print('Realizando previsões',end=' ')
#previsão para nosso conjunto de teste
y_prev = reglog.predict(x_teste_norm)

#Para o conjunto de dados de treino
y_prev_treino = reglog.predict(x_treino_norm)
print('[ok]')

#### Avaliando o modelo

In [ ]:
# exibindo a importância de cada variável no modelo preditivo
plt.rcParams['figure.figsize'] = [10.00,16.00]
plt.rcParams['figure.autolayout'] = True
#Importâncias para gráfico geral
importancias_rl = reglog.coef_[0]
importances = pd.Series(data=reglog.coef_[0], index=x.columns)
importances = importances.sort_values(ascending = False)
sns.barplot(x=importances, y=importances.index, orient='h').set_title('Importância de cada variável')
plt.show()

In [ ]:
#Matriz de confusão
cm = confusion_matrix(y_test,y_prev)
print(cm)

#### Calculando o G-Mean

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_prev).ravel()

# Calculando sensibilidade e especificidade
sensitivity = tp / (tp + fn)  # Recall para a classe positiva
specificity = tn / (tn + fp)  # Recall para a classe negativa

# Calculando o G-Mean
g_mean = np.sqrt(sensitivity * specificity)

print(f"Sensibilidade: {sensitivity:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"G-Mean: {g_mean:.4f}")

In [ ]:
#Configure...
portugues = True
nome_algoritmo = 'Logistic Regression'

#Outra opção de matriz de confusão
pd.options.display.float_format = '{:.2f}'.format

plt.rcParams["figure.figsize"] = [5.00,5.00]
plt.rcParams["figure.autolayout"] = True

f, ax = plt.subplots(figsize=(5,5))

if portugues:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Curta","Longa"],yticklabels=["Curta","Longa"])
  ax.set_xlabel('Valor predito')
  ax.set_ylabel('Valor real')
  #titulo = 'Sobrevida longa (1) ou curta (0) '+nome_algoritmo
  titulo = nome_algoritmo
  plt.title(titulo)
else:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Short","Long"],yticklabels=["Short","Long"])
  ax.set_xlabel('Predicted value')
  ax.set_ylabel('True value')
  titulo = 'Predictions of Long (1) or Short (0) Survival '+nome_algoritmo
  plt.title(titulo)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tick_params(axis='both',which='major',labelsize=10, labelbottom = False, top = True, labeltop = True)
plt.show()

In [ ]:

# Relatórios de classificação
relatorio_treino = classification_report(y_treino, y_prev_treino, output_dict=True)
relatorio_teste = classification_report(y_test, y_prev, output_dict=True)

# Transformar em DataFrames
df_treino = pd.DataFrame(relatorio_treino).transpose()
df_teste = pd.DataFrame(relatorio_teste).transpose()

# Remover a linha de 'accuracy' do corpo principal para exibi-la separadamente
acuracia_treino = df_treino.loc['accuracy', 'f1-score']
acuracia_teste = df_teste.loc['accuracy', 'f1-score']

df_treino = df_treino.drop('accuracy')
df_teste = df_teste.drop('accuracy')

# Exibir os resultados formatados
print('=== Resultados para Logistic Regression ===\n')

print('Relatório de Classificação - Treino')
print(f'Acurácia: {acuracia_treino:.4f}')
display(df_treino.style.format(precision=4).background_gradient(cmap='Blues'))

print('\nRelatório de Classificação - Teste')
print(f'Acurácia: {acuracia_teste:.4f}')
display(df_teste.style.format(precision=4).background_gradient(cmap='Greens'))

print('[ok]')

In [ ]:
# Probabilidades previstas para a classe positiva (Classe 1)
y_pred_proba = reglog.predict_proba(x_teste_norm)[:, 1]
# Obter valores de FPR (False Positive Rate), TPR (True Positive Rate) e thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Calcular a área sob a curva (AUC)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {auc_score:.4f}")
#Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label="Random Guess")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curva ROC Logistic Regression")
plt.legend(loc="lower right")
plt.grid()
plt.show()

### 6.3. Treinando o modelo KNN

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Supomos que você já tem X_train, X_test, y_train, y_test

# Defina o modelo base
knn = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [3, 5, 7, 10, 15, 20],       # Inclua mais vizinhos
    'weights': ['uniform', 'distance'],         # Ponderação dos vizinhos
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],  # Algoritmos disponíveis
    'p': [1, 2, 3, 4],                          # Inclua mais valores de p (distância Minkowski)
    'leaf_size': [10, 20, 30, 50, 100],         # Tamanhos de folha para algoritmos baseados em árvores
    'metric': ['minkowski', 'euclidean', 'manhattan', 'chebyshev']  # Métricas de distância adicionais
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=knn,             # Modelo base
    param_grid=param_grid,     # Hiperparâmetros expandidos
    scoring='f1_weighted',     # Métrica a ser otimizada
    cv=5,                      # Validação cruzada com 5 folds
    verbose=2,                 # Nível de detalhamento na saída
    n_jobs=-1                  # Usa todos os núcleos disponíveis
)

# Executa a busca
grid_search.fit(x_treino_norm, y_treino)

# Melhor combinação de parâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor pontuação obtida
print("Melhor pontuação de validação cruzada:", grid_search.best_score_)

# Avalie o modelo no conjunto de teste
best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_teste_norm)

print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred))

In [ ]:
print('Treinando o knn ...', end='')

# n_neighbors=1 valor de k

# Ajustando o modelo com os melhores parâmetros
knn = KNeighborsClassifier(
    algorithm='auto',   # Seleção automática do melhor algoritmo
    leaf_size=10,
    metric='minkowski',
    n_neighbors=15,      # Número de vizinhos
    p=1,                # Distância de Manhattan (p=1)
    weights='distance'  # Pesos baseados na distância
)

#Treinando
knn.fit(x_treino_norm,y_treino)
print('[ok]')

#### Realizando previsões

In [ ]:
print('Realizando previsões ...', end='')
previsoes = knn.predict(x_teste_norm)

previsoes_treino = knn.predict(x_treino_norm)
print('[ok]')

In [ ]:
#Acurácia (dados de teste x o que foi previsto pelo modelo)
acuracia = accuracy_score(y_test,previsoes)
print(acuracia*100)

In [ ]:
from sklearn.inspection import permutation_importance
# exibindo a importância de cada variável no modelo preditivo
plt.rcParams['figure.figsize'] = [10.00,16.00]
plt.rcParams['figure.autolayout'] = True
perm_importance = permutation_importance(knn, x_teste_norm, y_test, n_repeats=30, random_state=42)
# Converter importâncias para uma série do Pandas
importances = pd.Series(perm_importance.importances_mean, index=x.columns)
#importâncias para o gráfico geral
importancias_knn = importances
# Plotar as importâncias das variáveis
plt.figure(figsize=(10, 16))
importances.sort_values().plot(kind='barh')
plt.xlabel('Importância')
plt.ylabel('Variáveis')
plt.title('Importância das Variáveis no Classificador KNN (após pré-processamento)')
plt.show()

#### Matriz de confusão

In [ ]:
#Matriz de confusão
cm = confusion_matrix(y_test,previsoes)
print(cm)

#### Calculando o G-Mean

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, previsoes).ravel()

# Calculando sensibilidade e especificidade
sensitivity = tp / (tp + fn)  # Recall para a classe positiva
specificity = tn / (tn + fp)  # Recall para a classe negativa

# Calculando o G-Mean
g_mean = np.sqrt(sensitivity * specificity)

print(f"Sensibilidade: {sensitivity:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"G-Mean: {g_mean:.4f}")

In [ ]:
#Configure...
portugues = True
nome_algoritmo = 'K Nearest Neighbor'

#Outra opção de matriz de confusão
pd.options.display.float_format = '{:.2f}'.format

plt.rcParams["figure.figsize"] = [5.00,5.00]
plt.rcParams["figure.autolayout"] = True

f, ax = plt.subplots(figsize=(5,5))

if portugues:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Curta","Longa"],yticklabels=["Curta","Longa"])
  ax.set_xlabel('Valor predito')
  ax.set_ylabel('Valor real')
  #titulo = 'Previsões de sobrevida longa (1) ou curta (0) '+nome_algoritmo
  titulo = nome_algoritmo
  plt.title(titulo)
else:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Short","Long"],yticklabels=["Short","Long"])
  ax.set_xlabel('Predicted value')
  ax.set_ylabel('True value')
  titulo = 'Predictions of Long (1) or Short (0) Survival '+nome_algoritmo
  plt.title(titulo)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tick_params(axis='both',which='major',labelsize=10, labelbottom = False, top = True, labeltop = True)
plt.show()

In [ ]:
# Relatórios de classificação
relatorio_treino = classification_report(y_treino, previsoes_treino, output_dict=True)
relatorio_teste = classification_report(y_test, previsoes, output_dict=True)

# Transformar em DataFrames
df_treino = pd.DataFrame(relatorio_treino).transpose()
df_teste = pd.DataFrame(relatorio_teste).transpose()

# Remover a linha de 'accuracy' do corpo principal para exibi-la separadamente
acuracia_treino = df_treino.loc['accuracy', 'f1-score']
acuracia_teste = df_teste.loc['accuracy', 'f1-score']

df_treino = df_treino.drop('accuracy')
df_teste = df_teste.drop('accuracy')

# Exibir os resultados formatados
print('=== Resultados para KNN ===\n')

print('Relatório de Classificação - Treino')
print(f'Acurácia: {acuracia_treino:.4f}')
display(df_treino.style.format(precision=4).background_gradient(cmap='Blues'))

print('\nRelatório de Classificação - Teste')
print(f'Acurácia: {acuracia_teste:.4f}')
display(df_teste.style.format(precision=4).background_gradient(cmap='Greens'))

print('[ok]')

In [ ]:
# Probabilidades previstas para a classe positiva (Classe 1)
y_pred_proba = knn.predict_proba(x_teste_norm)[:, 1]
# Obter valores de FPR (False Positive Rate), TPR (True Positive Rate) e thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Calcular a área sob a curva (AUC)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {auc_score:.4f}")
#Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label="Random Guess")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curva ROC KNN")
plt.legend(loc="lower right")
plt.grid()
plt.show()

### 6.4. Treinando o modelo Árvore de decisão


In [ ]:
# Otimizando os parâmetros

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Supomos que você já tem X_train, X_test, y_train, y_test

# Defina o modelo base
dt = DecisionTreeClassifier(random_state=42)

param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],  # Critérios de divisão
    'max_depth': [None, 3, 5, 10, 15, 20, 30],     # Profundidade máxima da árvore
    'min_samples_split': [2, 5, 10, 20],           # Número mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4, 10],             # Número mínimo de amostras em uma folha
    'max_features': [None, 'sqrt', 'log2', 0.5],   # Máximo de características consideradas para divisão
    'min_impurity_decrease': [0.0, 0.01, 0.1],     # Mínimo de impureza para dividir
    'max_leaf_nodes': [None, 10, 20, 50, 100],     # Limite máximo de nós folha
    'min_weight_fraction_leaf': [0.0, 0.01, 0.1]   # Fração mínima de peso de amostras em uma folha
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=dt,                # Modelo base
    param_grid=param_grid,       # Conjunto de hiperparâmetros
    scoring='f1_weighted',       # Métrica a ser otimizada
    cv=5,                        # Validação cruzada com 5 folds
    verbose=2,                   # Nível de detalhamento na saída
    n_jobs=-1                    # Usa todos os núcleos disponíveis
)

# Executa a busca
grid_search.fit(x_treino_norm, y_treino)

# Melhor combinação de parâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor pontuação obtida
print("Melhor pontuação de validação cruzada:", grid_search.best_score_)

# Avalie o modelo no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_tunning = best_model.predict(x_teste_norm)

print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred_tunning))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report

# Modelo base
dt = DecisionTreeClassifier(random_state=42)

# Espaço de hiperparâmetros para busca
param_distributions = {
    'criterion': ['gini', 'entropy'],               # Critério para medir a qualidade da divisão
    'max_depth': [None, 10, 20, 30, 40, 50],        # Profundidade máxima da árvore
    'min_samples_split': [2, 5, 10, 20],            # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 5, 10],              # Mínimo de amostras por folha
    'max_features': [None, 'sqrt', 'log2'],         # Número máximo de características para considerar em uma divisão
    'ccp_alpha': [0.0, 0.01, 0.05, 0.1, 0.5],       # Parâmetro de poda
}

# Configurando o RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=dt,                      # Modelo base
    param_distributions=param_distributions,  # Hiperparâmetros
    n_iter=100,                        # Número de combinações aleatórias a testar
    scoring='f1_weighted',             # Métrica de avaliação
    cv=5,                              # Validação cruzada com 5 folds
    random_state=42,                   # Para reprodutibilidade
    n_jobs=-1,                         # Usar todos os núcleos disponíveis
    verbose=2                          # Nível de detalhamento
)

# Executa o RandomizedSearchCV
random_search.fit(x_treino_norm, y_treino)

# Exibe os melhores hiperparâmetros encontrados
print("Melhores parâmetros encontrados:")
print(random_search.best_params_)

# Melhor modelo
best_model = random_search.best_estimator_

# Avaliar no conjunto de teste
y_pred_tunning = best_model.predict(x_teste_norm)
print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred_tunning))

In [ ]:
from sklearn.model_selection import cross_val_score

# Mais ajustes de parâmetros
# Modelo inicial (ajustado sem poda)
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(x_treino_norm, y_treino)

# Obter o caminho de poda
path = dt.cost_complexity_pruning_path(x_treino_norm, y_treino)
ccp_alphas = path.ccp_alphas  # Valores de ccp_alpha disponíveis
impurities = path.impurities  # Impurezas totais para cada alpha

# Visualizar os valores de ccp_alpha
print("Valores de ccp_alpha disponíveis:", ccp_alphas)



# Testar diferentes valores de ccp_alpha
ccp_alphas = path.ccp_alphas
scores = []

for ccp_alpha in ccp_alphas:
    dt = DecisionTreeClassifier(random_state=42, class_weight='balanced', ccp_alpha=ccp_alpha)
    cv_scores = cross_val_score(dt, x_treino_norm, y_treino, cv=5, scoring='f1_weighted')
    scores.append(cv_scores.mean())

# Visualizar a relação entre ccp_alpha e pontuação
plt.figure(figsize=(10, 6))
plt.plot(ccp_alphas, scores, marker='o')
plt.xlabel("ccp_alpha")
plt.ylabel("F1-Score Médio (Cross-Validation)")
plt.title("Ajuste de ccp_alpha para o DecisionTreeClassifier")
plt.show()

# Melhor ccp_alpha
best_alpha = ccp_alphas[scores.index(max(scores))]
print("Melhor valor de ccp_alpha:", best_alpha)

In [ ]:
# Criando o modelo DecisionTreeClassifier

# Ajustando o modelo com os melhores parâmetros encontrados
dt = DecisionTreeClassifier(
    criterion='entropy',      # Critério para divisão dos nós (entropia)
    max_depth=None,           # Sem limite para a profundidade da árvore
    max_features=None,        # Considera todas as características para dividir
    max_leaf_nodes = 100,
    min_impurity_decrease = 0.0,
    min_weight_fraction_leaf = 0.0,
    min_samples_leaf=1,       # Mínimo de 1 amostra em cada nó folha
    min_samples_split=2,      # Mínimo de 2 amostras para dividir um nó interno
    random_state=42,          # Define a semente para reprodutibilidade
    #class_weight='balanced',   # Ajusta pesos automaticamente para lidar com classes desbalanceadas
    ccp_alpha=best_alpha 
)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

#modelo = RandomForestClassifier(random_state=42)
rfe = RFE(estimator=dt, n_features_to_select=10)
rfe.fit(x_treino_norm, y_treino)
variaveis_selecionadas = x.columns[rfe.support_]
print("Variáveis Selecionadas:", list(variaveis_selecionadas))

In [ ]:
# Treinando o modelo
dt.fit(x_treino_norm, y_treino)

In [ ]:
# Fazendo previsões
y_pred_dt = dt.predict(x_teste_norm)

# Para os dados de treino
y_pred_dt_treino = dt.predict(x_treino_norm)

In [ ]:
#Acurácia (dados de teste x o que foi previsto pelo modelo)
acuracia = accuracy_score(y_test,y_pred_dt)
print(acuracia*100)

In [ ]:
# exibindo a importância de cada variável no modelo preditivo
plt.rcParams['figure.figsize'] = [10.00,16.00]
plt.rcParams['figure.autolayout'] = True
importances = pd.Series(data=dt.feature_importances_, index=x.columns)
#importâncias para o gráfico geral
importancias_dt = importances
importances = importances.sort_values(ascending = False)
sns.barplot(x=importances, y=importances.index, orient='h').set_title('Importância de cada variável')
plt.show()

In [ ]:
#Matriz de confusão
cm = confusion_matrix(y_test,y_pred_dt)
print(cm)

#### Calculando o G-Mean

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_dt).ravel()

# Calculando sensibilidade e especificidade
sensitivity = tp / (tp + fn)  # Recall para a classe positiva
specificity = tn / (tn + fp)  # Recall para a classe negativa

# Calculando o G-Mean
g_mean = np.sqrt(sensitivity * specificity)

print(f"Sensibilidade: {sensitivity:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"G-Mean: {g_mean:.4f}")

In [ ]:
#Configure...
portugues = True
nome_algoritmo = 'Decision Tree'

#Outra opção de matriz de confusão
pd.options.display.float_format = '{:.2f}'.format

plt.rcParams["figure.figsize"] = [5.00,5.00]
plt.rcParams["figure.autolayout"] = True

f, ax = plt.subplots(figsize=(5,5))

if portugues:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Curta","Longa"],yticklabels=["Curta","Longa"])
  ax.set_xlabel('Valor predito')
  ax.set_ylabel('Valor real')
  #titulo = 'Previsões de sobrevida longa (1) ou curta (0) '+nome_algoritmo
  titulo = nome_algoritmo
  plt.title(titulo)
else:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Short","Long"],yticklabels=["Short","Long"])
  ax.set_xlabel('Predicted value')
  ax.set_ylabel('True value')
  titulo = 'Predictions of Long (1) or Short (0) Survival '+nome_algoritmo
  plt.title(titulo)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tick_params(axis='both',which='major', labelbottom = False, top = True, labeltop = True)
plt.show()

In [ ]:
# #Calculado métricas de avaliação do modelo

# Relatórios de classificação
relatorio_treino = classification_report(y_treino, y_pred_dt_treino, output_dict=True)
relatorio_teste = classification_report(y_test, y_pred_dt, output_dict=True)

# Transformar em DataFrames
df_treino = pd.DataFrame(relatorio_treino).transpose()
df_teste = pd.DataFrame(relatorio_teste).transpose()

# Remover a linha de 'accuracy' do corpo principal para exibi-la separadamente
acuracia_treino = df_treino.loc['accuracy', 'f1-score']
acuracia_teste = df_teste.loc['accuracy', 'f1-score']

df_treino = df_treino.drop('accuracy')
df_teste = df_teste.drop('accuracy')

# Exibir os resultados formatados
print('=== Resultados para Decision Tree Classifier ===\n')

print('Relatório de Classificação - Treino')
print(f'Acurácia: {acuracia_treino:.4f}')
display(df_treino.style.format(precision=4).background_gradient(cmap='Blues'))

print('\nRelatório de Classificação - Teste')
print(f'Acurácia: {acuracia_teste:.4f}')
display(df_teste.style.format(precision=4).background_gradient(cmap='Greens'))

print('[ok]')

In [ ]:
# Probabilidades previstas para a classe positiva (Classe 1)
y_pred_proba = dt.predict_proba(x_teste_norm)[:, 1]
# Obter valores de FPR (False Positive Rate), TPR (True Positive Rate) e thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Calcular a área sob a curva (AUC)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {auc_score:.4f}")
#Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label="Random Guess")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curva ROC Decision Tree Classifier")
plt.legend(loc="lower right")
plt.grid()
plt.show()

### 6.5. Teste com o SVM (SVC)

In [ ]:
#Melhores parâmetros para SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Supomos que você já tem X_train, X_test, y_train, y_test

# Defina o modelo base
svc = SVC(probability=True, random_state=42)

# Defina os parâmetros para busca
param_grid = {
    'C': [0.1, 1, 10, 100],         # Parâmetro de regularização
    'kernel': ['linear', 'rbf'],    # Tipos de kernel
    'gamma': ['scale', 'auto', 0.1, 1],  # Coeficiente do kernel para 'rbf' e 'poly'
    'class_weight': [None, 'balanced']  # Lida com classes desbalanceadas
}

# Configurando o GridSearchCV
grid_search = GridSearchCV(
    estimator=svc,              # Modelo base
    param_grid=param_grid,      # Conjunto de hiperparâmetros
    scoring='f1_weighted',      # Métrica a ser otimizada
    cv=5,                       # Validação cruzada com 5 folds
    verbose=2,                  # Nível de detalhamento na saída
    n_jobs=-1                   # Usa todos os núcleos disponíveis
)

# Executa a busca
grid_search.fit(x_treino_norm, y_treino)

# Melhor combinação de parâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor pontuação obtida
print("Melhor pontuação de validação cruzada:", grid_search.best_score_)

# Avalie o modelo no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_tunning = best_model.predict(x_teste_norm)

print("\nRelatório de classificação no conjunto de teste:")
print(classification_report(y_test, y_pred_tunning))

In [ ]:
# Criar um modelo SVM com kernel RBF

from sklearn.svm import SVC

# Ajustando o modelo com os melhores parâmetros encontrados
svm = SVC(
    kernel='rbf',         # Kernel radial básico
    C=100,                # Parâmetro de regularização
    gamma=1,        # Coeficiente para o kernel
    probability=True,     # Permite prever probabilidades (necessário para métricas como ROC-AUC)
    class_weight=None     # Sem ajuste de peso automático para classes
)


In [ ]:
# Treinar o modelo
svm.fit(x_treino_norm, y_treino)

In [ ]:
# Fazendo previsões
y_pred_svm = svm.predict(x_teste_norm)
# Com os dados de treinamento
y_pred_svm_treino = svm.predict(x_treino_norm)

In [ ]:
#Calculando a acurácia
accuracy = accuracy_score(y_test, y_pred_svm)
print(accuracy*100)

In [ ]:
# Calcular a importância das características usando permutation importance
result = permutation_importance(svm, x_teste_norm, y_test, n_repeats=10, random_state=42)

# Obter as importâncias médias e os desvios padrão
importances_mean = result.importances_mean

# Obter os nomes das características
feature_names = x.columns

# Ajustar os parâmetros do matplotlib
plt.rcParams['figure.figsize'] = [10.00, 16.00]
plt.rcParams['figure.autolayout'] = True

# Criar uma série pandas para as importâncias e ordenar
importances = pd.Series(data=importances_mean, index=feature_names)
importances = importances.sort_values(ascending=False)
#importâncias para o gráfico geral
importancias_svm = importances
# Plotar o gráfico de barras usando seaborn
sns.barplot(x=importances, y=importances.index, orient='h').set_title('Importância de cada variável')
plt.xlabel('Importância da Característica')
plt.ylabel('Características')
plt.show()

In [ ]:
#Matriz de confusão
cm = confusion_matrix(y_test,y_pred_svm)
print(cm)

#### Calculando o G-Mean

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_svm).ravel()

# Calculando sensibilidade e especificidade
sensitivity = tp / (tp + fn)  # Recall para a classe positiva
specificity = tn / (tn + fp)  # Recall para a classe negativa

# Calculando o G-Mean
g_mean = np.sqrt(sensitivity * specificity)

print(f"Sensibilidade: {sensitivity:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"G-Mean: {g_mean:.4f}")

In [ ]:
#Configure...
portugues = True
nome_algoritmo = 'Support Vector Classifier'

#Outra opção de matriz de confusão
pd.options.display.float_format = '{:.2f}'.format

plt.rcParams["figure.figsize"] = [5.00,5.00]
plt.rcParams["figure.autolayout"] = True

f, ax = plt.subplots(figsize=(5,5))

if portugues:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Curta","Longa"],yticklabels=["Curta","Longa"])
  ax.set_xlabel('Valor predito')
  ax.set_ylabel('Valor real')
  #titulo = 'Previsões de sobrevida longa (1) ou curta (0) '+nome_algoritmo
  titulo = nome_algoritmo
  plt.title(titulo)
else:
  sns.heatmap(cm,cmap='Blues',annot=True, cbar_kws={'orientation':'vertical'}, fmt='.0f',xticklabels=["Short","Long"],yticklabels=["Short","Long"])
  ax.set_xlabel('Predicted value')
  ax.set_ylabel('True value')
  titulo = 'Predictions of Long (1) or Short (0) Survival '+nome_algoritmo
  plt.title(titulo)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tick_params(axis='both',which='major',labelsize=10, labelbottom = False, top = True, labeltop = True)
plt.show()

In [ ]:
# Relatórios de classificação
relatorio_treino = classification_report(y_treino, y_pred_svm_treino, output_dict=True)
relatorio_teste = classification_report(y_test, y_pred_svm, output_dict=True)

# Transformar em DataFrames
df_treino = pd.DataFrame(relatorio_treino).transpose()
df_teste = pd.DataFrame(relatorio_teste).transpose()

# Remover a linha de 'accuracy' do corpo principal para exibi-la separadamente
acuracia_treino = df_treino.loc['accuracy', 'f1-score']
acuracia_teste = df_teste.loc['accuracy', 'f1-score']

df_treino = df_treino.drop('accuracy')
df_teste = df_teste.drop('accuracy')

# Exibir os resultados formatados
print('=== Resultados para SVC ===\n')

print('Relatório de Classificação - Treino')
print(f'Acurácia: {acuracia_treino:.4f}')
display(df_treino.style.format(precision=4).background_gradient(cmap='Blues'))

print('\nRelatório de Classificação - Teste')
print(f'Acurácia: {acuracia_teste:.4f}')
display(df_teste.style.format(precision=4).background_gradient(cmap='Greens'))

print('[ok]')

In [ ]:
# Probabilidades previstas para a classe positiva (Classe 1)
y_pred_proba = svm.predict_proba(x_teste_norm)[:, 1]
# Obter valores de FPR (False Positive Rate), TPR (True Positive Rate) e thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Calcular a área sob a curva (AUC)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {auc_score:.4f}")
#Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f"ROC Curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label="Random Guess")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")
plt.title("Curva ROC SVC")
plt.legend(loc="lower right")
plt.grid()
plt.show()

# 7. Curva de aprendizado

### Interpretaçãol do gráfico

###### **Overfitting**: Se a curva de treino tem uma alta acurácia, mas a curva de validação/teste tem uma acurácia baixa, isso é um sinal de overfitting.
###### **Underfitting**: Se ambas as curvas têm uma acurácia baixa, o modelo pode estar subajustado.
###### **Bom ajuste**: As curvas de treino e validação/teste devem convergir, indicando que o modelo está generalizando bem.

*   RF
*   LR
*   KNN
*   DT
*   SVM (SVC)


### RandomForest

In [ ]:
#Curva de aprendizado para o random forest
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
# Gerando a curva de aprendizado
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    estimator=clf,
    X=x_treino_norm,
    y=y_treino,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=-1  # Paralelização
)

# Calculando médias e desvios padrão
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotando a curva de aprendizado
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Acurácia no Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Acurácia na Validação")
plt.fill_between(train_sizes,
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Pontuação de Acurácia")
plt.title("Curva de Aprendizado - random forest com Dados Balanceados")
plt.legend(loc="best")
plt.grid()
plt.show()

### Regressão logísitica

In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
# Gerando a curva de aprendizado
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    estimator=reglog,
    X=x_treino_norm,
    y=y_treino,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=-1  # Paralelização
)

# Calculando médias e desvios padrão
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotando a curva de aprendizado
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Acurácia no Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Acurácia na Validação")
plt.fill_between(train_sizes,
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Pontuação de Acurácia")
plt.title("Curva de Aprendizado - SVM com Dados Balanceados")
plt.legend(loc="best")
plt.grid()
plt.show()

### KNN

In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
# Gerando a curva de aprendizado
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    estimator=knn,
    X=x_treino_norm,
    y=y_treino,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=-1  # Paralelização
)

# Calculando médias e desvios padrão
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotando a curva de aprendizado
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Acurácia no Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Acurácia na Validação")
plt.fill_between(train_sizes,
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Pontuação de Acurácia")
plt.title("Curva de Aprendizado - SVM com Dados Balanceados")
plt.legend(loc="best")
plt.grid()
plt.show()

### Árvore de decisão

In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
# Gerando a curva de aprendizado
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    estimator=dt,
    X=x_treino_norm,
    y=y_treino,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=-1  # Paralelização
)

# Calculando médias e desvios padrão
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotando a curva de aprendizado
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Acurácia no Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Acurácia na Validação")
plt.fill_between(train_sizes,
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Pontuação de Acurácia")
plt.title("Curva de Aprendizado - SVM com Dados Balanceados")
plt.legend(loc="best")
plt.grid()
plt.show()

### SVM



In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
# Gerando a curva de aprendizado
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
train_sizes, train_scores, test_scores = learning_curve(
    estimator=svm,
    X=x_treino_norm,
    y=y_treino,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring='accuracy',
    n_jobs=-1  # Paralelização
)

# Calculando médias e desvios padrão
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotando a curva de aprendizado
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Acurácia no Treinamento")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Acurácia na Validação")
plt.fill_between(train_sizes,
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std,
                 alpha=0.1, color="r")
plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.1, color="g")
plt.xlabel("Tamanho do Conjunto de Treinamento")
plt.ylabel("Pontuação de Acurácia")
plt.title("Curva de Aprendizado - SVM com Dados Balanceados")
plt.legend(loc="best")
plt.grid()
plt.show()

# 7. Validação cruzada

In [ ]:
#Confogirações iniciais
n_divisoes = 10

In [ ]:
def validacao_cruzada(modelo, X, y, oversampling=False, folds=5):
  if isinstance(X, np.ndarray):
      X = pd.DataFrame(X)
  if isinstance(y, np.ndarray):
      y = pd.DataFrame(y)

  #Kfold do scikit-Learn
  kfold = KFold(n_splits=folds, shuffle=True)

  #Lista de acurácias de cada split
  acuracias_split = []

  #Iterar sobre os splits
  for idx,(idx_treino, idx_validacao) in enumerate(kfold.split(X)):
    X_split_treino = X.iloc[idx_treino,:]
    # Changed line: Use iloc with only one dimension for y
    y_split_treino = y.iloc[idx_treino] #<- Changed this line
    #Aplicar apenas a oversampling so no split de treinamento
    if oversampling:
      sm = SMOTE(random_state=42)
      X_split_treino, y_split_treino = sm.fit_resample(X_split_treino, y_split_treino)

    #Treinar os modelos com a parcela de dados balanaceda
    modelo.fit(X_split_treino, y_split_treino.values.flatten())

    #Separando o conjunto de validação
    X_split_validacao = X.iloc[idx_validacao,:]
    # Changed line: Use iloc with only one dimension for y
    y_split_validacao = y.iloc[idx_validacao] #<- Changed this line

    #Realizando previsões com o conjunto de teste
    predicao = modelo.predict(X_split_validacao)

    #Calcular a acurácia para o split atual
    acuracia_split = accuracy_score(y_split_validacao, predicao)

    #Adicionar a lista
    acuracias_split.append(acuracia_split)

    print(f'Acurácia do split {idx}: {acuracia_split:.4f}')

  #Calcular a acurácia média
  acuracia_media = np.mean(acuracias_split)
  print(f'Acurácia média {acuracia_media:.4f}')

  return acuracia_split

### Random Forest

In [ ]:
print('Validação cruzada Random Forest não usando oversampling ')
print(validacao_cruzada(clf, x_treino_norm_n_balanceado, y_treino_n_balanceado, oversampling=False, folds=n_divisoes))

### Regressão Logística

In [ ]:
print('Validação cruzada Regressão Logística não usando oversampling ')
print(validacao_cruzada(reglog, x_treino_norm_n_balanceado, y_treino_n_balanceado, oversampling=False, folds=n_divisoes))

### Regressão Logística

In [ ]:
print('Validação cruzada KNN não usando oversampling ')
print(validacao_cruzada(knn, x_treino_norm_n_balanceado, y_treino_n_balanceado, oversampling=False, folds=n_divisoes))

### Árvore de decisão

In [ ]:
print('Validação cruzada Decicion Tree não usando oversampling ')
print(validacao_cruzada(dt, x_treino_norm_n_balanceado, y_treino_n_balanceado, oversampling=False, folds=n_divisoes))

### SVC

In [ ]:
print('Validação cruzada  Support Vector Classifier não usando oversampling ')
print(validacao_cruzada(svm, x_treino_norm_n_balanceado, y_treino_n_balanceado, oversampling=False, folds=n_divisoes))

# 8. Shap


In [ ]:
# Importar o módulo shap
import shap

# Nomes das colunas
feature_names = x.columns

# Explicador para RandomForestClassifier
explainer_rf = shap.Explainer(clf.predict_proba, x_teste_norm , feature_names=feature_names)
# Explicador para DecisionTreeClassifier
explainer_dt = shap.Explainer(dt.predict_proba, x_teste_norm , feature_names=feature_names)
# Explicador para LogisticRegression
explainer_lr = shap.Explainer(reglog.predict_proba, x_teste_norm , feature_names=feature_names)
# Explicador para KNN
explainer_knn = shap.Explainer(knn.predict_proba, x_teste_norm , feature_names=feature_names)
# Explicador para SVC (SVM)
explainer_svc = shap.Explainer(svm.predict_proba, x_teste_norm , feature_names=feature_names)

# Calculando os valores de SHAP para cada modelo
# RandomForestClassifier
shap_values_rf = explainer_rf(x_teste_norm)
# DecisionTreeClassifier
shap_values_dt = explainer_dt(x_teste_norm)
# LogisticRegression
shap_values_lr = explainer_lr(x_teste_norm)
# KNN
shap_values_knn = explainer_knn(x_teste_norm)
# SVC
shap_values_svc = explainer_svc(x_teste_norm)

#Obtém SHAP values para a classe 1 RandomForestClassifier
shap_values_rf_class1 = shap_values_rf[..., 1]
#Obtém SHAP values para a classe 1 DecisionTreeClassifier
shap_values_dt_class1 = shap_values_dt[..., 1]
#Obtém SHAP values para a classe 1 LogisticRegression
shap_values_lr_class1 = shap_values_lr[..., 1]
#Obtém SHAP values para a classe 1 KNN
shap_values_knn_class1 = shap_values_knn[..., 1]
#Obtém SHAP values para a classe 1 SVC
shap_values_svc_class1 = shap_values_svc[..., 1]




In [ ]:
print(x_teste_norm.shape)  # Número de linhas e colunas do dataset
print(len(feature_names))  # Comprimento da lista de nomes das features

#Obtém SHAP values para a classe 1 RandomForestClassifier
shap_values_rf_class1 = shap_values_rf[..., 1]
#Obtém SHAP values para a classe 1 DecisionTreeClassifier
shap_values_dt_class1 = shap_values_dt[..., 1]
#Obtém SHAP values para a classe 1 LogisticRegression
shap_values_lr_class1 = shap_values_lr[..., 1]
#Obtém SHAP values para a classe 1 KNN
shap_values_knn_class1 = shap_values_knn[..., 1]
#Obtém SHAP values para a classe 1 SVC
shap_values_svc_class1 = shap_values_svc[..., 1]

print("Shape de x_teste_norm:", x_teste_norm.shape)
print("Shape de shap_values_rf[:, 1]:", shap_values_rf[:, 1].shape)

print("Shape de x_teste_norm:", x_teste_norm.shape)
print("Shape de shap_values_rf_class1:", shap_values_rf_class1.shape)


In [ ]:
# Summary plot 
shap.summary_plot(shap_values_rf_class1, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho e o título do gráfico
plt.title("Random Forest")

# Exibir o gráfico ajustado
plt.show()

In [ ]:
shap_values_rf_class0 = shap_values_rf[..., 0]
# Summary plot 
shap.summary_plot(shap_values_rf_class0, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho e o título do gráfico
plt.title("Random Forest")

# Exibir o gráfico ajustado
plt.show()

In [ ]:
# Gerar o summary plot sem exibir
shap.summary_plot(shap_values_rf_class1, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho do gráfico
plt.title("SHAP Summary Plot for Random Forest")

# Aumentar o tamanho da fonte das features no eixo Y
ax = plt.gca()  # Obter o eixo atual
ax.tick_params(axis='y',)  # Ajustar o tamanho da fonte dos rótulos do eixo Y

# Exibir o gráfico ajustado
plt.show()

In [ ]:
# Violin plot
shap.plots.violin(shap_values_rf_class1, features=x_teste_norm, feature_names=feature_names, plot_type="layered_violin", show=False, max_display=4)
plt.title("Random Forest")
plt.show()

In [ ]:
# importância das variáveis
shap.plots.bar(shap_values_rf_class1, show=False)
plt.title("Random Forest")
plt.show()

In [ ]:
# Para a primeira observação no conjunto de teste
shap.waterfall_plot(shap_values_rf_class1[0])

In [ ]:
# Explica uma instância específica (índice)
shap.plots.waterfall(shap_values_rf_class1[3])

In [ ]:
shap.summary_plot(shap_values_rf_class1, x_teste_norm, plot_type="bar", show=False)
plt.title("Random Forest summary plot barra")
plt.show()

In [ ]:
#shap.plots.bar(shap_values_rf, clustering=clust, clustering_cutoff=1)
#shap.plots.scatter(shap_values_rf, ylabel="SHAP value\n(higher means more likely to renew)")
# clust = shap.utils.hclust(x_teste_norm, y_test, linkage="single")
# shap.plots.bar(shap_values_rf, clustering=clust, clustering_cutoff=1)

In [ ]:
shap.plots.heatmap(shap_values_rf_class1, show=False)
plt.title("Heatmap Random Forest")
plt.gcf().set_size_inches(25, 25)
plt.show()

In [ ]:
#Obtém SHAP values para a classe 1
shap_values_dt_class1 = shap_values_dt[..., 1]

shap.summary_plot(shap_values_dt_class1, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho e o título do gráfico
plt.title("Decision Tree Classifier")
# Exibir o gráfico ajustado
plt.show()

In [ ]:
# importância das variáveis
shap.plots.bar(shap_values_dt_class1, show=False)
plt.title("Decision Tree")
plt.show()

In [ ]:
# Explica uma instância específica (índice)
shap.plots.waterfall(shap_values_dt_class1[3])

In [ ]:
shap.summary_plot(shap_values_dt_class1, x_teste_norm, plot_type="bar", show=False)
plt.title("Decision Tree Classifier summary barra")
plt.show()

In [ ]:
shap.plots.heatmap(shap_values_dt_class1, show=False)
plt.title("Heatmap Decision Tree Classifier")
plt.gcf().set_size_inches(25, 25)
plt.show()

In [ ]:
shap.summary_plot(shap_values_lr_class1, x_teste_norm, show=False, max_display=5)

plt.title("Logistic Regression")
# Ajustar o tamanho e o título do gráfico
plt.title("Logistic Regression")  # Define o título e o tamanho da fonte

# Exibir o gráfico ajustado
plt.show()

In [ ]:
# Violin plot
shap.plots.violin(shap_values_lr_class1, features=x_teste_norm, feature_names=feature_names, plot_type="layered_violin", show=False)
plt.title("Logistic Regression")
plt.show()

In [ ]:
# importância das variáveis
shap.plots.bar(shap_values_lr_class1, show=False)
plt.title("Logistic Regression")
plt.show()

In [ ]:
# Explica uma instância específica (índice)
shap.plots.waterfall(shap_values_lr_class1[3])

In [ ]:
shap.summary_plot(shap_values_lr_class1, x_teste_norm, plot_type="bar", show=False)
plt.title("Logistic Regression summary barra")
plt.show()

In [ ]:
shap.plots.heatmap(shap_values_lr_class1, show=False)
plt.title("Heatmap Logistic Regression")
plt.gcf().set_size_inches(25, 25)
plt.show()

In [ ]:
shap.summary_plot(shap_values_knn_class1, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho e o título do gráfico
plt.title("K Nearest Neighbor")  # Define o título e o tamanho da fonte

# Exibir o gráfico ajustado
plt.show()

In [ ]:
# Violin plot
shap.plots.violin(shap_values_knn_class1, features=x_teste_norm, feature_names=feature_names, plot_type="layered_violin", show=False)
plt.title("KNN")
plt.show()

In [ ]:
# importância das variáveis
shap.plots.bar(shap_values_knn_class1, show=False)
plt.title("K Nearest Neighbor")
plt.show()

In [ ]:
# Explica uma instância específica (índice)
shap.plots.waterfall(shap_values_knn_class1[3])

In [ ]:
shap.summary_plot(shap_values_knn_class1, x_teste_norm, plot_type="bar", show=False)
plt.title("KNN summary barra")
plt.show()

In [ ]:
shap.plots.heatmap(shap_values_knn_class1, show=False)
plt.title("Heatmap K Nearest Neighbor")
plt.gcf().set_size_inches(25, 25)
plt.show()

In [ ]:
shap.summary_plot(shap_values_svc_class1, x_teste_norm, show=False, max_display=5)

# Ajustar o tamanho e o título do gráfico
plt.title("Support Vector Classifier")  # Define o título e o tamanho da fonte

# Exibir o gráfico ajustado
plt.show()

In [ ]:
# importância das variáveis
shap.plots.bar(shap_values_svc_class1, show=False)
plt.title("Support Vector Classifier")
plt.show()

In [ ]:
# Explica uma instância específica (índice)
shap.plots.waterfall(shap_values_svc_class1[3])

In [ ]:
shap.summary_plot(shap_values_svc_class1, x_teste_norm, plot_type="bar", show=False)
plt.title("SVC summary barra")
plt.show()

In [ ]:
shap.plots.heatmap(shap_values_svc_class1, show=False)
plt.title("Heatmap Support Vector Classifier")
plt.gcf().set_size_inches(25, 25)
plt.show()

# 9 Curva de aprendizado

In [ ]:
from sklearn.metrics import roc_curve, auc

portugues = True

# Inicializar os classificadores
classifiers = {
    "RF": clf,
    "LR": reglog,
    "DT": dt,
    "KNN": knn,
    "SVM": svm  # SVC com probabilidade para calcular ROC
}

# Plotar a Curva ROC para cada classificador
plt.figure(figsize=(10, 8))

for name, modelo in classifiers.items():
    # Treinar o modelo
    modelo.fit(x_treino_norm, y_treino)
    
    # Obter as probabilidades de predição para a classe positiva
    y_prob = modelo.predict_proba(x_teste_norm)[:, 1]
    
    # Calcular a curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    
    # Calcular a AUC
    roc_auc = auc(fpr, tpr)
    
    # Plotar a curva ROC
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

# Plotar a linha diagonal (modelo aleatório)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')

if portugues:
    # Definir título e rótulos
    plt.title('Curvas ROC')
    plt.xlabel('Taxa de Falsos Positivos')
    plt.ylabel('Taxa de Verdadeiros Positivos')
else:
    plt.title('ROC Curves')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

plt.legend(loc="lower right")

# Exibir o gráfico
plt.show()